# Adaptive HRC Experiment Suite

Notebook interface for the clean `src.experiments` runner. The suite is organized by logical experiment names rather than numbered tests, with separate main and appendix registries. Each run writes one `result.json` per seed folder, one `aggregate.json` per experiment, flat PNG figures, and a root `run.json` with progress/status metadata under `eval_results/`.


In [1]:
import importlib
import os
import sys

# Run this notebook with the Jupyter/VS Code kernel named `venv`.
_EXPECTED_PREFIX = "/home/abd/Documents/project/venv"
if os.path.abspath(sys.prefix) != _EXPECTED_PREFIX: raise RuntimeError(f"Select the Jupyter kernel named `venv` before running HRC.ipynb. Current executable: {sys.executable}")

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-hrc")
print(f"Kernel executable: {sys.executable}")

from src.environment import StateTracker, gen
from src import experiments

print(f"{StateTracker().n_features} raw state features")
print(f"Default seeds: {tuple(experiments.DEFAULT_SEEDS)}")
print(f"Recipe catalog entries: {len(gen.recipe_library())}")
print("Experiment module: src.experiments")

for recipe_name in gen.recipe_library(): print(recipe_name)


Kernel executable: /home/abd/Documents/project/venv/bin/python
470 raw state features
Default seeds: (1337, 2024, 7, 9001, 31415)
Recipe catalog entries: 30
Experiment module: src.experiments
tomato_onion_soup_v1
tomato_soup
mushroom_soup
seasoned_mixture_soup
grilled_steak
burger
seasoned_chicken
garlic_fish
simple_salad
grated_cheese_salad
smoothie
yoghurt_smoothie
boiled_eggs
boiled_rice
tomato_garlic_soup
mushroom_garlic_soup
tomato_mushroom_soup
onion_rice_pot
tomato_cheese_salad
tomato_lettuce_salad
banana_strawberry_fruit_bowl
oil_tomato_salad
scrambled_eggs_pan
mushroom_omelette
meat_mushroom_skillet
garlic_chicken_salad
chicken_rice_plate
fish_rice_plate
yoghurt_fruit_bowl
rice_mushroom_bowl


## Run the experiment suite

In [2]:
import src.models as _hrc_models
import src.memory as _hrc_memory
import src.adaptive_agent as _hrc_adaptive_agent
import src.baselines as _hrc_baselines

_hrc_models         = importlib.reload(_hrc_models)
_hrc_memory         = importlib.reload(_hrc_memory)
_hrc_adaptive_agent = importlib.reload(_hrc_adaptive_agent)
_hrc_baselines      = importlib.reload(_hrc_baselines)
experiments         = importlib.reload(experiments)

run_config = experiments.RunConfig(
    seeds=experiments.DEFAULT_SEEDS,
    output_root="eval_results",
    timestamp_subdir=True,
    workers=len(experiments.DEFAULT_SEEDS),  # one process per seed so each experiment finishes its seed batch together
    baselines=experiments.PAPER_BASELINES,
    quick=False,
    profile=False,
    log_full_distributions=False,
    valid_action_expansion=False,  # keep legacy fast MaxEnt action support unless explicitly stress-testing valid-action expansion
)

suite_config = experiments.ExperimentSuiteConfig(
    deployment_stream=experiments.DeploymentStreamConfig(n_recipes=10, n_users=4, n_phase_b_events=120),
    cross_recipe_transfer=experiments.TransferSuiteConfig(n_recipes=7, n_preferences=7, diagonal_cycles=1, offdiag_repeats=2),
    decay_reentry=experiments.DecayReentrySuiteConfig(n_recipes=12, gap_sweep=(5, 10, 15, 30, 45)),
    disambiguation_audit=experiments.DisambiguationAuditConfig(n_recipes=12),
    materiality_preflight=experiments.MaterialityPreflightConfig(n_recipes=15, n_preferences=7),
)

# Publication harness: five canonical scenarios plus appendix sweeps by default.
# Use experiments.MAIN_EXPERIMENTS only for the strict canonical run.
experiments_to_run = experiments.MAIN_EXPERIMENTS + experiments.APPENDIX_EXPERIMENTS
# experiments_to_run = experiments.MAIN_EXPERIMENTS

print(f"Planned experiments: {len(experiments_to_run)} -> {experiments_to_run}")
print(f"Seeds: {run_config.seeds}")
print(f"Workers: {run_config.workers or len(run_config.seeds)}")
print("Progress will stream below as each seed finishes. ETA is batch-aware: seeds run in parallel, and future experiments use per-experiment historical runtimes when available.")

result = experiments.run_suite(
    run_config=run_config,
    suite_config=suite_config,
    experiments=experiments_to_run,
)


Planned experiments: 8 -> ('materiality_preflight', 'deployment_stream', 'cross_recipe_transfer', 'decay_reentry', 'disambiguation_audit', 'zipf_usage_sweep', 'cl_regularizer_comparison', 'posterior_ablation_matrix')
Seeds: (1337, 2024, 7, 9001, 31415)
Workers: 5
Progress and ETA will stream below as each seed finishes.
[materiality_preflight] start: 5 seeds, workers=5
  [materiality_preflight] seed 1337 done in 0.2s (1/40, elapsed=0.2s ETA=9.4s)
  [materiality_preflight] seed 2024 done in 0.2s (2/40, elapsed=0.2s ETA=4.6s)
  [materiality_preflight] seed 31415 done in 0.2s (3/40, elapsed=0.3s ETA=3.3s)
  [materiality_preflight] seed 7 done in 0.3s (4/40, elapsed=0.3s ETA=2.4s)
  [materiality_preflight] seed 9001 done in 0.3s (5/40, elapsed=0.3s ETA=1.9s)
[materiality_preflight] completed
[deployment_stream] start: 5 seeds, workers=5
  [deployment_stream] seed 1337 done in 5069.2s (6/40, elapsed=5071.1s ETA=28736.1s)
  [deployment_stream] seed 2024 done in 5265.2s (7/40, elapsed=5267.2s

In [3]:
artifact_root = result.get("out_dir") or result.get("run_dir")
if artifact_root is None:
    raise RuntimeError("No output directory found in `result`. Run the experiment cell first.")
print("Completed experiments:")
for experiment_id in result.get("completed_experiments", []):
    print(f"  {experiment_id}")
if result.get("partial_experiments"):
    print("Partial experiments:")
    for experiment_id in result.get("partial_experiments", []):
        print(f"  {experiment_id}")
if result.get("failed_experiments"):
    print("Failed experiments:")
    for key, err in result["failed_experiments"].items():
        print(f"  {key}: {err}")
print()
print(f"Run JSON: {os.path.join(artifact_root, 'run.json')}")


Completed experiments:
  materiality_preflight
  deployment_stream
  cross_recipe_transfer
  decay_reentry
  disambiguation_audit
  zipf_usage_sweep
  cl_regularizer_comparison
  posterior_ablation_matrix

Run JSON: eval_results/2026-05-19_1401/run.json


In [4]:
from pathlib import Path

artifact_root = result.get("out_dir") or result.get("run_dir")
if artifact_root is None:
    raise RuntimeError("No output directory found in `result`. Run the experiment cell first.")
artifact_root = Path(artifact_root)
experiments.render_figures(artifact_root)
print("Artifacts written to:")
print(f"  {artifact_root}")
print()
for path in sorted(artifact_root.rglob("*")):
    if path.is_file() and path.suffix in {".json", ".png"}:
        print(path)


Artifacts written to:
  eval_results/2026-05-19_1401

eval_results/2026-05-19_1401/aggregate/run.json
eval_results/2026-05-19_1401/cl_regularizer_comparison/aggregate/aggregate.json
eval_results/2026-05-19_1401/cl_regularizer_comparison/aggregate/aggregate_F2_four_cell_grouped_bar.png
eval_results/2026-05-19_1401/cl_regularizer_comparison/aggregate/aggregate_F2_four_cell_heatmap.png
eval_results/2026-05-19_1401/cl_regularizer_comparison/aggregate/aggregate_F2_seen_recipe_preference.png
eval_results/2026-05-19_1401/cl_regularizer_comparison/aggregate/aggregate_F3_forgetting_curve.png
eval_results/2026-05-19_1401/cl_regularizer_comparison/aggregate/aggregate_F4_per_step_adaptation.png
eval_results/2026-05-19_1401/cl_regularizer_comparison/aggregate/aggregate_F5_coverage_accuracy.png
eval_results/2026-05-19_1401/cl_regularizer_comparison/aggregate/aggregate_adaptation_latency_steps.png
eval_results/2026-05-19_1401/cl_regularizer_comparison/aggregate/aggregate_assistance_coverage.png
eval_